
## 01MIAR - Actividad Video Valencia Pollution - Code

*Ivan Fuertes*

## Web API pública
- Interfaz de programación que se pone a disposición de cualquier desarrollador para acceder a datos o funcionalidades
### Valencia Open Data
- https://opendata.vlci.valencia.es/
#### Estaciones contaminación atmosférica
- https://opendata.vlci.valencia.es/dataset/estacions-contaminacio-atmosferiques-estaciones-contaminacion-atmosfericas

#### API Creada con CKAN
- https://docs.ckan.org/en/2.11/

### Requests
- Librería para hacer peticiones Http a web
- https://requests.readthedocs.io/en/latest/

In [21]:
import requests

In [ ]:
# function to get a json response from a url + params

def make_request(base_url, params):
    response = requests.get(base_url, params = params)
    if response:
        return response.json()
    
    else:
        print(response)
        raise Exception("Error Downloading JSON")
    
print("datos json ",make_request)

datos json  <function make_request at 0x0000019C5218E5E0>


- URL Base del servidor
- https://geoportal.valencia.es/server/rest/services/OPENDATA/

In [8]:
server_url = "https://geoportal.valencia.es/server/rest/services/OPENDATA/"

- Prueba de la llamada a la API
- https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query?where=1=1&outFields=*&f=json

In [9]:
base_url = server_url + "https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query?where=1=1&outFields=*&f=json"

#### Parámetros del Query
- `'f' = 'json'` - formato del fichero de respuesta
- `'outFields' = '*'` - campos en la respuesta, todos
- `'where' = '1=1'` - filtro

In [10]:
params = {'f' : 'json', 'outFields' : '*', 'where' : '1=1'}
print("base url",base_url)
print("datos  params",params)

base url https://geoportal.valencia.es/server/rest/services/OPENDATA/https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query?where=1=1&outFields=*&f=json
datos  params {'f': 'json', 'outFields': '*', 'where': '1=1'}


#### Realizar petición y explorar resultados
- `'fieldAliases'` = nombres de campos y sus alias
- `'fields'` = nombres de campos, alias, tipos y longitudes
- `'features'` = lista de estaciones y lecturas
  - `'attributes'` = lectura de una estación
  - `'geometry'` = coordenadas geométricas de una estación (sistema proyectado UTM)

In [11]:
import json   # to acquire and format json string

response = make_request("https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query?where=1=1&outFields=*&f=json", params)

print(json.dumps(response, indent = 2))

{
  "displayFieldName": "nombre",
  "fieldAliases": {
    "objectid": "objectid",
    "nombre": "Nom / Nombre",
    "direccion": "Adre\u00e7a / Direccion",
    "tipozona": "Tipus Zona / Tipo Zona",
    "parametros": "Par\u00e0metres / Par\u00e1metros",
    "mediciones": "Mesuraments / Mediciones",
    "so2": "so2",
    "no2": "no2",
    "o3": "o3",
    "co": "co",
    "pm10": "pm10",
    "pm25": "pm25",
    "tipoemisio": "tipoemision",
    "fecha_carg": "fecha_carga",
    "calidad_am": "calidad_ambiental",
    "fiwareid": "fiwareid"
  },
  "geometryType": "esriGeometryPoint",
  "spatialReference": {
    "wkid": 25830,
    "latestWkid": 25830
  },
  "fields": [
    {
      "name": "objectid",
      "type": "esriFieldTypeOID",
      "alias": "objectid"
    },
    {
      "name": "nombre",
      "type": "esriFieldTypeString",
      "alias": "Nom / Nombre",
      "length": 254
    },
    {
      "name": "direccion",
      "type": "esriFieldTypeString",
      "alias": "Adre\u00e7a / Direcci

In [12]:
from pyproj import Transformer   # to converto from UTM to GPS lat+long
import pandas as pd

transformer = Transformer.from_crs("EPSG:32630", "EPSG:4326", always_xy=True)

df = pd.DataFrame()

for feature in response['features']:
    serie = pd.Series(feature['attributes'])

    # transform from UTM to GPS
    serie["lon"], serie["lat"] = transformer.transform(feature['geometry']['x'], feature['geometry']['y'])
    
    df = pd.concat([df, serie.to_frame().T], axis = 0, ignore_index = True)

# format datetime from epoch (as miliseconds) to timestamp (GMT)
df['fecha_carg'] = pd.to_datetime(df['fecha_carg'], unit = 'ms')

# Convertir columnas de contaminantes y coordenadas de object a float para permitir calculos estadisticos
numeric_cols = ['so2', 'no2', 'o3', 'co', 'pm10', 'pm25', 'lon', 'lat']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')


In [13]:
display(df.head(3))

,objectid,nombre,direccion,tipozona,parametros,mediciones,so2,no2,o3,co,pm10,pm25,tipoemisio,fecha_carg,calidad_am,fiwareid,lon,lat
0,20,Cabanyal,CABANYAL,Urbana,None,None,NaN,19.0,NaN,NaN,32.0,11.0,Fondo,2026-05-29 17:00:00,Razonablemente Buena,A09_CABANYAL_60m,-0.328535,39.474391
1,21,Olivereta,OLIVERETA,Urbana,"Óxidos de nitrógeno totales (NOx),Monóxido de ...",None,NaN,36.0,NaN,NaN,48.0,16.0,Tráfico,2026-05-29 17:00:00,Regular,A10_OLIVERETA_60m,-0.405923,39.469244
2,14,Boulevar Sur,BULEVARD SUD,Urbana,"Dióxido de azufre (SO2),Ozono,Óxidos de nitróg...",http://mapas.valencia.es/WebsMunicipales/uploa...,2.0,9.0,104.0,NaN,NaN,NaN,Tráfico,2026-05-29 17:00:00,Regular,A02_BULEVARDSUD_60m,-0.396338,39.450396


### Tomar muestras de los datos cada hora durante un día completo

In [14]:
sleep_time = 60 * 60 # 60 minutes in seconds
total_time = (24 * 60 * 60) + sleep_time  # 1 day to take samples (plus one sleep extra just in case)
current_time = 0 # current time before ending sampling

In [15]:
# endpoints
server_url = "https://geoportal.valencia.es/server/rest/services/OPENDATA/"
base_url = server_url + "MedioAmbiente/MapServer/156/query"

In [16]:
# params
params = {'f' : 'json', 'outFields' : '*', 'where' : '1=1'}

In [ ]:
# path to save dataset in csv format
import os
from time import sleep

path_csv = ['res', 'valencia_pollution_dataset.csv']
path_csv_solved = os.path.join(*path_csv)
first_time = True # flag for one use only to save headers in csv

# Crear el directorio de salida si no existe para evitar OSError al guardar el CSV
os.makedirs('res', exist_ok=True)


In [18]:
# main loop
while current_time < total_time:
    # Usar base_url definida en la celda de endpoints en lugar de URL hardcodeada
    pollution_list = make_request(base_url, params)['features']

    print(f"Current Time = {current_time}, Records Processed = {len(pollution_list)}")
    
    df = pd.DataFrame()
    for feature in pollution_list:
        serie = pd.Series(feature['attributes'])
        serie["lon"], serie["lat"] = transformer.transform(feature['geometry']['x'], feature['geometry']['y'])
        df = pd.concat([df, serie.to_frame().T], axis = 0, ignore_index = True)
        
    df['fecha_carg'] = pd.to_datetime(df['fecha_carg'], unit = 'ms')

    # Convertir columnas numericas para que el CSV guarde tipos correctos
    numeric_cols = ['so2', 'no2', 'o3', 'co', 'pm10', 'pm25', 'lon', 'lat']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

    df.to_csv(path_csv_solved, sep=',', header=first_time, mode='a', index=False)

    first_time = False
    sleep(sleep_time)
    current_time += sleep_time


Current Time = 0, Records Processed = 11


KeyboardInterrupt: 

In [ ]:
display(df.head(5))

,objectid,nombre,direccion,tipozona,parametros,mediciones,so2,no2,o3,co,pm10,pm25,tipoemisio,fecha_carg,calidad_am,fiwareid,lon,lat
0,20,Cabanyal,CABANYAL,Urbana,None,None,None,19.0,None,None,32.0,11.0,Fondo,2026-05-29 17:00:00,Razonablemente Buena,A09_CABANYAL_60m,-0.328535,39.474391
1,21,Olivereta,OLIVERETA,Urbana,"Óxidos de nitrógeno totales (NOx),Monóxido de ...",None,None,36.0,None,None,48.0,16.0,Tráfico,2026-05-29 17:00:00,Regular,A10_OLIVERETA_60m,-0.405923,39.469244
2,14,Boulevar Sur,BULEVARD SUD,Urbana,"Dióxido de azufre (SO2),Ozono,Óxidos de nitróg...",http://mapas.valencia.es/WebsMunicipales/uploa...,2.0,9.0,104.0,None,None,None,Tráfico,2026-05-29 17:00:00,Regular,A02_BULEVARDSUD_60m,-0.396338,39.450396
3,15,Molí del Sol,MOLÍ DEL SOL,Suburbana,"Dióxido de azufre (SO2),Monóxido de carbono (C...",http://mapas.valencia.es/WebsMunicipales/uploa...,5.0,15.0,105.0,0.0,37.0,10.0,Tráfico,2026-05-29 17:00:00,Regular,A03_MOLISOL_60m,-0.40881,39.481112
4,17,Universidad Politécnica,POLITÈCNIC,Suburbana,"Dióxido de azufre (SO2),Ozono,Óxidos de nitróg...",http://mapas.valencia.es/WebsMunicipales/uploa...,0.0,25.0,90.0,None,18.0,9.0,Fondo,2026-05-29 17:00:00,Razonablemente Buena,A05_POLITECNIC_60m,-0.337401,39.479644


In [19]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   objectid    11 non-null     object        
 1   nombre      11 non-null     object        
 2   direccion   11 non-null     object        
 3   tipozona    11 non-null     object        
 4   parametros  10 non-null     object        
 5   mediciones  7 non-null      object        
 6   so2         6 non-null      float64       
 7   no2         11 non-null     float64       
 8   o3          6 non-null      float64       
 9   co          3 non-null      float64       
 10  pm10        9 non-null      float64       
 11  pm25        9 non-null      float64       
 12  tipoemisio  11 non-null     object        
 13  fecha_carg  11 non-null     datetime64[ns]
 14  calidad_am  11 non-null     object        
 15  fiwareid    11 non-null     object        
 16  lon         11 non-null     

None

In [20]:
display(df.describe(include='all'))

,objectid,nombre,direccion,tipozona,parametros,mediciones,so2,no2,o3,co,pm10,pm25,tipoemisio,fecha_carg,calidad_am,fiwareid,lon,lat
count,11.0,11,11,11,10,7,6.000000,11.000000,6.000000,3.0,9.000000,9.000000,11,11,11,11,11.000000,11.000000
unique,11.0,11,11,2,8,7,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,2,11,NaN,NaN
top,20.0,Cabanyal,CABANYAL,Urbana,"Óxidos de nitrógeno totales (NOx),Monóxido de ...",http://mapas.valencia.es/WebsMunicipales/uploa...,NaN,NaN,NaN,NaN,NaN,NaN,Tráfico,NaN,Razonablemente Buena,A09_CABANYAL_60m,NaN,NaN
freq,1.0,1,1,9,3,1,NaN,NaN,NaN,NaN,NaN,NaN,8,NaN,6,1,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,3.833333,27.909091,89.833333,0.0,35.111111,11.333333,NaN,2026-05-29 17:00:00,NaN,NaN,-0.370217,39.467885
min,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,9.000000,52.000000,0.0,18.000000,8.000000,NaN,2026-05-29 17:00:00,NaN,NaN,-0.408810,39.450396
25%,NaN,NaN,NaN,NaN,NaN,NaN,2.500000,17.000000,72.750000,0.0,32.000000,10.000000,NaN,2026-05-29 17:00:00,NaN,NaN,-0.398874,39.458625
50%,NaN,NaN,NaN,NaN,NaN,NaN,4.500000,24.000000,97.000000,0.0,37.000000,11.000000,NaN,2026-05-29 17:00:00,NaN,NaN,-0.376398,39.469244
75%,NaN,NaN,NaN,NaN,NaN,NaN,5.000000,37.000000,104.750000,0.0,37.000000,13.000000,NaN,2026-05-29 17:00:00,NaN,NaN,-0.340193,39.477016
max,NaN,NaN,NaN,NaN,NaN,NaN,7.000000,54.000000,121.000000,0.0,48.000000,16.000000,NaN,2026-05-29 17:00:00,NaN,NaN,-0.328289,39.481112
